# 🛍️ RAG Retail Agent — Google Colab Demo
**Project:** RAG-based Product Q&A Agent | **Stack:** LangChain + FAISS + HuggingFace + FastAPI

This notebook lets you run the full RAG pipeline **without any API keys or GPU** using free HuggingFace models.

### Architecture
```
User Question → FastAPI Endpoint → LangChain RAG Chain
                                        ↓
                              FAISS Vector Search
                                        ↓
                          HuggingFace Embeddings (MiniLM)
                                        ↓
                          flan-t5-base LLM → Answer
```

In [ ]:
# STEP 1: Install dependencies
!pip install langchain langchain-community faiss-cpu sentence-transformers transformers -q

In [ ]:
# STEP 2: Define product catalog
import json

products = [
  {"id": "P001", "name": "Nike Air Max 270", "category": "Shoes", "price": 149.99, "stock": 35,
   "description": "Lightweight running shoe with Air Max cushioning. Available in black, white, red. Sizes 38-47. Water-resistant."},
  {"id": "P002", "name": "Adidas Ultraboost 22", "category": "Shoes", "price": 189.99, "stock": 20,
   "description": "High-performance running shoe with Boost midsole. Grey and blue. Sizes 39-46. Best for long-distance running."},
  {"id": "P003", "name": "Samsung Galaxy S25", "category": "Electronics", "price": 999.00, "stock": 12,
   "description": "Flagship Android smartphone. 6.7 inch AMOLED, 200MP camera, 5G. 256GB storage."},
  {"id": "P004", "name": "Apple AirPods Pro 3", "category": "Electronics", "price": 279.00, "stock": 50,
   "description": "Wireless earbuds with ANC. 30 hours battery. Compatible with iPhone and Android."},
  {"id": "P005", "name": "Levi 501 Jeans", "category": "Clothing", "price": 89.99, "stock": 80,
   "description": "Classic straight-fit denim. Dark blue, light blue, black. Sizes 28-40."},
  {"id": "P009", "name": "Whey Protein Powder", "category": "Sports", "price": 49.99, "stock": 100,
   "description": "25g protein per serving. 30 servings. Vanilla flavour. Post-workout recovery."},
  {"id": "P010", "name": "Yoga Mat 6mm", "category": "Sports", "price": 34.99, "stock": 60,
   "description": "Non-slip yoga mat. 6mm thickness. 183x61cm. Purple, blue, black. Includes strap."}
]
print(f'Loaded {len(products)} products')

In [ ]:
# STEP 3: Build FAISS Vector Index
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Convert products to Documents
docs = []
for p in products:
    text = f"Product: {p['name']}\nCategory: {p['category']}\nPrice: €{p['price']}\nStock: {p['stock']} units\nDescription: {p['description']}"
    docs.append(Document(page_content=text, metadata={"id": p["id"], "name": p["name"]}))

# Load free embedding model (downloads ~80MB once)
print('Loading embedding model... (first run downloads ~80MB)')
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# Build FAISS index
vectorstore = FAISS.from_documents(docs, embeddings)
print(f'FAISS index built with {vectorstore.index.ntotal} vectors')

In [ ]:
# STEP 4: Load free LLM + build RAG chain
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from transformers import pipeline

print('Loading LLM (flan-t5-base ~250MB)...')
generator = pipeline('text2text-generation', model='google/flan-t5-base', max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=generator)

prompt_template = """You are a helpful retail assistant.
Use the product information below to answer the customer's question.

Product Context:
{context}

Customer Question: {question}
Helpful Answer:"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=['context', 'question'])
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
qa_chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type='stuff', retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={'prompt': PROMPT}
)
print('RAG chain ready!')

In [ ]:
# STEP 5: Ask questions!
import time

def ask(question):
    print(f'\n❓ Question: {question}')
    t0 = time.time()
    result = qa_chain({'query': question})
    latency = (time.time() - t0) * 1000
    print(f'🤖 Answer: {result["result"].strip()}')
    sources = list(set([d.metadata['name'] for d in result['source_documents']]))
    print(f'📦 Sources: {sources}')
    print(f'⚡ Latency: {latency:.0f}ms')

# Try it out
ask('Do you have running shoes under 200 euros?')
ask('What is good for post-workout recovery?')
ask('Tell me about the Samsung Galaxy')

In [ ]:
# STEP 6: Test the FastAPI backend locally using requests
# First, start the backend in a separate terminal:
#   cd backend && uvicorn main:app --reload
# Then run this cell:

import requests

# Uncomment to test live API:
# r = requests.post('http://localhost:8000/ask', json={'question': 'What running shoes do you have?'})
# print(r.json())

print('To test the API:')
print('1. Run: uvicorn backend.main:app --reload')
print('2. Visit: http://localhost:8000/docs (Swagger UI)')
print('3. Or run: streamlit run frontend/app.py')